# Cloud Data Pipeline: S3 + EC2 + PostgreSQL


## System Architecture

API -> EC2 (ETL) -> S3 (Data Lake) -> PostgreSQL (Analytics)

Flow:

1. Extract data from API
2. Save raw data to S3
3. Transform data
4. Save processed data to S3
5. Load into PostgreSQL
6. Run analytics queries

👉 This is real production-style architecture


## Extract Data

```python
import requests

def extract():
    url = "https://jsonplaceholder.typicode.com/users"
    return requests.get(url).json()
```


## Save RAW to S3

```python
import boto3
import json
from datetime import datetime

s3 = boto3.client("s3")
BUCKET = "your-data-engineering-bucket"

def save_raw(data):
    filename = f"raw/users_{datetime.now().isoformat()}.json"

    with open("raw.json", "w") as f:
        json.dump(data, f)

    s3.upload_file("raw.json", BUCKET, filename)
    print("Raw data saved to S3:", filename)
```


## Transform Data

```python
def transform(data):
    return [
        {
            "id": d["id"],
            "name": d["name"].title(),
            "email": d["email"].lower()
        }
        for d in data
    ]
```


## Save Processed to S3

```python
def save_processed(data):
    filename = f"processed/users_{datetime.now().isoformat()}.json"

    with open("processed.json", "w") as f:
        json.dump(data, f)

    s3.upload_file("processed.json", BUCKET, filename)
    print("Processed data saved to S3:", filename)
```


## Load to PostgreSQL

```python
import psycopg2

def load(data):
    conn = psycopg2.connect(
        host="your-rds-endpoint",
        database="de_course",
        user="admin",
        password="admin"
    )

    cur = conn.cursor()

    for d in data:
        cur.execute("""
        INSERT INTO users (id, name, email)
        VALUES (%s, %s, %s)
        ON CONFLICT (id) DO NOTHING
        """, (d["id"], d["name"], d["email"]))

    conn.commit()
    conn.close()

    print("Loaded into PostgreSQL")
```


## Full Pipeline

```python
def run_pipeline():
    data = extract()

    save_raw(data)

    cleaned = transform(data)

    save_processed(cleaned)

    load(cleaned)

    print("Cloud pipeline executed successfully")

run_pipeline()
```


## Analytics Query

```python
conn = psycopg2.connect(
    host="your-rds-endpoint",
    database="de_course",
    user="admin",
    password="admin"
)

cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM users")
print(cur.fetchall())
```


## Practice

1. Run full pipeline on EC2
2. Check:
   - S3 raw folder
   - S3 processed folder
   - PostgreSQL table


## Assignment

Upgrade pipeline:

- Add:
  - logging
  - retry mechanism
  - incremental loading

Bonus:
- Save processed data as Parquet


## Interview Questions

1. How do you design a cloud data pipeline?
2. Why use S3 + RDS together?
3. How do you handle failures in cloud?
4. How do you scale pipelines?


## What You Just Built

This is the moment:

👉 You now have a cloud data platform

Data lake (S3)
Compute layer (EC2)
Database (RDS/Postgres)
Pipeline orchestration (you can plug Airflow)

👉 This is exactly what companies build.


---

**Next:** Cloud Project (Portfolio-Level System)

You will:
- Combine batch + streaming
- Add Airflow
- Add Kafka (optional)
- Build full platform

👉 This becomes your ultimate project


## Your tasks

- [ ] Run pipeline on EC2 end-to-end.
- [ ] Verify S3 raw data and S3 processed data paths.
- [ ] Verify DB data via SQL query.
- [ ] Add logging and retry behavior.
